# Feature Extraction: Full-Text Exploration

This notebook is the **first step** toward using StockTwits message *text* as a feature
source, going beyond the sentiment/volume/timing features in `features_01`-`features_05`
(all of which are computed from `merged_with_crsp_mlcrowd/`, a text-free source).

**Why this exists:** `FEATURE_IMPLEMENTATION_STATUS.md` marks Features 49-51 (Symbol Focus,
Co-mention Count, Broadcasting/Spam Ratio) and the Event-Category/Magnitude track as
**blocked — requires message text**. The raw StockTwits S3 export (see the downloader cell
in `data_cleaning.ipynb`) turns out to already include a `messages/` and `msg_info/` category
alongside `feature_wo_messages/` — they were downloaded but never wired into the pipeline.
This notebook confirms that data is usable, documents its quirks, and prototypes the text
processing building blocks before anything is productionized into a `features_06_*.ipynb`.

**Scope of this notebook:** exploratory only, on a single sample chunk. No full 15-year
processing and no pickle output here — see Section 6 for the roadmap.


## 1. Setup and Configuration

In [ ]:
import re
import ast
import html
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# =============================================================================
# DIRECTORY PATHS
# =============================================================================
DATA_DIR = Path(r"E:\Research_data\Stocktwits\dataset\v1\data\csv")

MESSAGES_DIR = DATA_DIR / "messages"                  # message_id, message_body
MSG_INFO_DIR = DATA_DIR / "msg_info"                  # message_id, length, important_words
FEATURE_WO_MESSAGES_DIR = DATA_DIR / "feature_wo_messages"  # message_id, user_id, created_at, sentiment, ...

# Sample files used for this exploration. NOTE: these three categories are NOT chunked
# the same way (205 files in messages/, 66 in msg_info/, 248 in feature_wo_messages/) --
# see Section 2. The "000" / "00" files happen to start at the same message_id, so they
# align well for a first sample, but file index alignment is not guaranteed in general.
SAMPLE_MESSAGES_FILE = MESSAGES_DIR / "msg_000.csv"
SAMPLE_MSG_INFO_FILE = MSG_INFO_DIR / "msg_info_00.csv"
SAMPLE_FEATURE_FILE = FEATURE_WO_MESSAGES_DIR / "feature_wo_messages_000.csv"

SAMPLE_NROWS = 200_000

print(f"Messages dir:   {MESSAGES_DIR} ({len(list(MESSAGES_DIR.glob('*.csv')))} files)")
print(f"Msg info dir:   {MSG_INFO_DIR} ({len(list(MSG_INFO_DIR.glob('*.csv')))} files)")
print(f"Feature dir:    {FEATURE_WO_MESSAGES_DIR} ({len(list(FEATURE_WO_MESSAGES_DIR.glob('*.csv')))} files)")


## 2. Inspect Raw Data Structure

The three sources share a `message_id` join key but are exported as **separate file sets
with different chunk counts** (205 / 66 / 248 files). That rules out joining by matching
file index in general -- the join must always be done on `message_id`. For this sample,
the first file in each set happens to start at `message_id == 4`, so we use them together.


In [ ]:
df_msgs_sample = pd.read_csv(SAMPLE_MESSAGES_FILE, nrows=SAMPLE_NROWS)
df_info_sample = pd.read_csv(SAMPLE_MSG_INFO_FILE, nrows=SAMPLE_NROWS)
df_feat_sample = pd.read_csv(SAMPLE_FEATURE_FILE, nrows=SAMPLE_NROWS)

print("messages/ columns:", list(df_msgs_sample.columns))
print(df_msgs_sample.head(3))
print()
print("msg_info/ columns:", list(df_info_sample.columns))
print(df_info_sample.head(3))
print()
print("feature_wo_messages/ columns:", list(df_feat_sample.columns))
print(df_feat_sample.head(3))


## 3. Data Quality: Malformed Rows

A small number of message bodies contain characters (stray quotes / embedded newlines)
that break naive CSV parsing, corrupting the `message_id` field of the following row.
This affects a negligible fraction of rows (~0.002% in this sample) -- we coerce
`message_id` to numeric and drop rows that fail to parse.


In [ ]:
df_msgs_sample['message_id'] = pd.to_numeric(df_msgs_sample['message_id'], errors='coerce')
n_bad = df_msgs_sample['message_id'].isna().sum()
print(f"Dropped {n_bad} malformed rows ({n_bad / len(df_msgs_sample):.4%} of sample)")

df_msgs_sample = df_msgs_sample.dropna(subset=['message_id']).copy()
df_msgs_sample['message_id'] = df_msgs_sample['message_id'].astype('int64')


## 4. Join Text with Sentiment/Metadata (Sample)

Inner-join the three sources on `message_id`. `symbol_list` and `important_words` are
stored as string-encoded Python lists and need `ast.literal_eval`. HTML entities
(e.g. `&#39;`) appear in raw message bodies and are unescaped for downstream NLP.


In [ ]:
df = (
    df_feat_sample
    .merge(df_msgs_sample, on='message_id', how='inner')
    .merge(df_info_sample, on='message_id', how='inner')
)

print(f"Joined {len(df):,} rows from {len(df_feat_sample):,} feature rows "
      f"({len(df) / len(df_feat_sample):.2%} coverage in this sample)")

df['symbol_list'] = df['symbol_list'].apply(ast.literal_eval)
df['important_words'] = df['important_words'].apply(ast.literal_eval)
df['message_body_clean'] = df['message_body'].apply(lambda t: html.unescape(str(t)))

df[['message_id', 'created_at', 'sentiment', 'symbol_list', 'important_words', 'message_body_clean']].head(10)


## 5. Text Cleaning Utilities

Regex-based extractors for cashtags, @-mentions, and URLs. These are the building
blocks for Features 49-51 (Symbol Focus, Co-mention Count, Broadcasting/Spam Ratio) and
for stripping boilerplate before any topic modeling / embedding work.


In [ ]:
CASHTAG_RE = re.compile(r'\$([A-Za-z]{1,6}(?:\.[A-Za-z]{1,2})?)\b')
MENTION_RE = re.compile(r'(?<!\w)@(\w{1,20})')
URL_RE = re.compile(r'https?://\S+')


def extract_cashtags(text):
    """Return list of tickers referenced via $CASHTAG syntax (upper-cased)."""
    return [t.upper() for t in CASHTAG_RE.findall(text)]


def extract_mentions(text):
    return MENTION_RE.findall(text)


def extract_urls(text):
    return URL_RE.findall(text)


def strip_boilerplate(text):
    """Remove cashtags, mentions, and URLs, leaving prose for topic modeling / embeddings."""
    text = URL_RE.sub(' ', text)
    text = CASHTAG_RE.sub(' ', text)
    text = MENTION_RE.sub(' ', text)
    return re.sub(r'\s+', ' ', text).strip()


df['cashtags'] = df['message_body_clean'].apply(extract_cashtags)
df['n_cashtags'] = df['cashtags'].apply(len)
df['mentions'] = df['message_body_clean'].apply(extract_mentions)
df['n_mentions'] = df['mentions'].apply(len)
df['n_urls'] = df['message_body_clean'].apply(lambda t: len(extract_urls(t)))
df['body_stripped'] = df['message_body_clean'].apply(strip_boilerplate)

df[['message_body_clean', 'cashtags', 'n_cashtags', 'n_mentions', 'n_urls']].head(10)


## 6. Exploratory Analysis

### 6.1 Message Length

In [ ]:
df['char_len'] = df['message_body_clean'].str.len()

print("Correlation between msg_info 'length' and recomputed char length:")
print(df[['length', 'char_len']].corr())

fig, ax = plt.subplots(figsize=(8, 4))
df['char_len'].plot(kind='hist', bins=60, ax=ax)
ax.set_xlabel('Message character length')
ax.set_title('Distribution of Message Length (sample)')
plt.tight_layout()
plt.show()


### 6.2 Cashtag Count -- groundwork for Feature 49 (Symbol Focus) and Feature 50 (Co-mention Count)

In [ ]:
print("Cashtags extracted via regex vs. symbol_list length (from feature_wo_messages):")
df['n_symbol_list'] = df['symbol_list'].apply(len)
print(df[['n_cashtags', 'n_symbol_list']].describe())

print()
print("Distribution of cashtag count per message:")
print(df['n_cashtags'].value_counts().sort_index().head(15))

# Feature 49 preview: Symbol Focus -- message mentions exactly one symbol
single_symbol_ratio = (df['n_cashtags'] == 1).mean()
print(f"\nShare of messages mentioning exactly 1 symbol (Symbol Focus proxy): {single_symbol_ratio:.2%}")

# Feature 50 preview: Co-mention Count -- avg number of OTHER symbols mentioned
comention_count = (df['n_cashtags'] - 1).clip(lower=0)
print(f"Mean co-mention count (Feature 50 proxy): {comention_count.mean():.2f}")


### 6.3 Broadcasting / Spam Proxy -- groundwork for Feature 51

In [ ]:
# Feature 51 preview: Broadcasting/Spam Ratio -- messages mentioning >5 other symbols
spam_ratio = (df['n_cashtags'] > 5).mean()
print(f"Share of messages mentioning >5 symbols (Broadcasting/Spam proxy): {spam_ratio:.2%}")

print(f"URL prevalence: {(df['n_urls'] > 0).mean():.2%} of messages contain a link")
print(f"Mention prevalence: {(df['n_mentions'] > 0).mean():.2%} of messages contain an @-mention")


### 6.4 Sentiment vs. Text Characteristics

In [ ]:
df_labeled = df[df['sentiment'].isin(['Bullish', 'Bearish'])]

print(df_labeled.groupby('sentiment')[['char_len', 'n_cashtags', 'n_mentions', 'n_urls']].mean())

fig, ax = plt.subplots(figsize=(6, 4))
df_labeled.boxplot(column='char_len', by='sentiment', ax=ax)
ax.set_title('Message Length by Sentiment')
plt.suptitle('')
plt.tight_layout()
plt.show()


### 6.5 Top `important_words` by Sentiment

A quick word-frequency comparison using the pre-extracted `important_words` column (from `msg_info/`) as a cheap bag-of-words, before investing in a full tokenization pipeline.

In [ ]:
from collections import Counter

def top_words(frame, n=20):
    counter = Counter()
    for words in frame['important_words']:
        counter.update(words)
    return counter.most_common(n)

print("Top words -- Bullish:")
print(top_words(df_labeled[df_labeled['sentiment'] == 'Bullish']))
print()
print("Top words -- Bearish:")
print(top_words(df_labeled[df_labeled['sentiment'] == 'Bearish']))


## 7. Roadmap: Full-Text Feature Engineering

This notebook only validates the data and prototypes primitives. Next steps, in
suggested order:

**A. Productionize the blocked features (small, well-scoped)**
1. `features_06_message_context.ipynb` -- Features 49-51 (Symbol Focus, Co-mention Count,
   Broadcasting/Spam Ratio) using `extract_cashtags()` above, aggregated to stock-day level.
2. `features_06_event_categories.ipynb` -- event classification (litigation, guidance,
   sales/contracts, etc.) and typed magnitude extraction (dollar/unit/percent/time amounts)
   via regex over `body_stripped`, per the reference design in the project notes
   (`event_classifier.py`, `event_magnitudes.py`).

**B. Broader NLP exploration (open-ended, per project scope)**
3. **Topic modeling** (NMF or LDA) over TF-IDF of `body_stripped` or the `important_words`
   bag-of-words, to discover latent conversation themes beyond hand-specified event
   categories.
4. **Sentence embeddings** for semantic clustering / similarity-based crowd-consensus
   features (e.g. how semantically dispersed is the day's conversation about a stock).
5. **Sentiment-lexicon validation**: compare the StockTwits self-reported `sentiment` tag
   against a text-derived polarity score to flag potentially mislabeled or noisy messages
   -- could sharpen every existing sentiment-based feature (01, 03, 05).
6. **Readability / complexity metrics** as a proxy for retail vs. informed posting style.

**C. Scaling to all 15 years (data engineering)**
`messages/` and `msg_info/` are NOT chunked the same way as `cleaned_by_year_mlcrowd/`
(different file counts, and as shown in Section 2, a single chunk can span many
calendar years because early StockTwits volume was very sparse). A full-history join
needs either:
- Build a `message_id -> year` lookup from `cleaned_by_year_mlcrowd/` (which already
  restricts to the 176M Bullish/Bearish-labeled messages), then stream through all
  `messages/` + `msg_info/` files once, routing each row to its output year bucket, or
- Skip messages that don't survive the Bull/Bear filter before any text processing,
  shrinking the workload from 501M to 176M messages up front.

No pickle output is produced by this notebook -- see `FEATURE_IMPLEMENTATION_STATUS.md`
for tracking once A.1/A.2 above are implemented and validated.
